In [3]:
import sys
import json
import random
import csv
import pandas as pd
import numpy as np
import os
from pathlib import Path
from sklearn.model_selection import train_test_split

import tarfile
import ijson

# Since I can't know the progress about how it's running, so I ask AI
# It told me could use tqdm to show the progress
from tqdm import tqdm 

# Add the project root to the Python path -- It has to to be done before everything
# 必须先把sys的目录改成项目根目录，否则后续的import会找不到项目内模块[例如config.py]
project_root = Path.cwd().parent.parent if Path.cwd().name == 'data' else Path.cwd().parent
sys.path.insert(0, str(project_root))

# Now import from project modules
from src.utils.filesUtil import save_json, save_csv, read_tar
from src.config import (
    TRAIN_IMG_DIR, TRAIN_ANNOTATION_FILE,
    TEST_IMG_DIR, TEST_ANNOTATION_FILE,
    DATA_SPLITS_DIR,
    # =============
    RANDOM_SEED,NUM_CLASSES,NUM_TRAIN_PER_CLASS,
    NUM_VAL_PER_CLASS,NUM_TEST_PER_CLASS,
    # =============
    TRAIN_IMG_DIR_TAR, TRAIN_ANNOTATION_TAR, VAL_TAR_PATH, VAL_JSON_TAR_PATH,
)
# ====================== 配置参数（可修改） ======================


# ==============================================================

# public config only for this py
TRAIN_TAR_PATH = TRAIN_IMG_DIR_TAR
TRAIN_JSON_TAR_PATH = TRAIN_ANNOTATION_TAR
OUTPUT_TAR_PATH = "tiny_inat_500.tar.gz"
OUTPUT_DIR = Path("./tiny_inat_500")  # 可选：同时解压到文件夹

## Waht in side ##

In [4]:
def find_out(path, max_preview=30):
    '''
    浏览TAR里面到底有点啥，看看结构
    看不出来，东西太多，所以换了个跳序
    tf.next() -> tf.fileobj.seek()
    从而跳过大量字节，而不是一点点读过去
    '''
    with tarfile.open(path, 'r:gz') as tf:
        # MY PCCCCCCCCCCCCCCCCCCCCCCC
        '''
        print(f"压缩包内总文件/文件夹数: {len(tf.getnames())}")
        print("前 10 个文件/文件夹路径:")
        for name in tf.getnames()[:10]:
            print(f"  - {name}")
        '''
        count = 0

        while True:
            member = tf.next()
            if member is None:
                print("已经读取到压缩包末尾。")
                break
                
            m_type = "[][]目录/List[][]" if member.isdir() else "File"

            m_size_mb = member.size / (1024 * 1024) # MB
            print(f"{m_type} {member.name} ({m_size_mb:.2f} MB)")

            count += 1
            if count >= max_preview:
                print(f"the {max_preview} files are been displaced.")
                break

In [5]:
def find_out(path, max_preview=30):
    with tarfile.open(path, 'r:gz') as tf:
        print("正在对 42GB 压缩包进行【跨目录跳跃式】快速采样...")
        print("-" * 60)
        
        seen_directories = set()
        
        while len(seen_directories) < max_preview:
            # 1. 使用底层方法读取下一个 TarInfo，这不会读取文件内容
            member = tf.next()
            if member is None:
                print("\n已扫描至压缩包末尾。")
                break
                
            # 2. 解析当前文件所属的子目录路径
            # 例如: 'train_mini/00328_Animalia/.../abc.jpg' -> 'train_mini/00328_Animalia_...'
            parts = member.name.split('/')
            if len(parts) >= 2:
                parent_dir = parts[1]  # 取到分类文件夹这一层
            else:
                parent_dir = member.name

            # 3. 如果是一个新文件夹，或者是顶层目录，打印出来
            if parent_dir not in seen_directories and member.isfile():
                seen_directories.add(parent_dir)
                m_size_mb = member.size / (1024 * 1024)
                print(f"New target [{len(seen_directories)}/{max_preview}]:")
                print(f"  V .../{parent_dir}/")
                print(f"  |____ Files /////: {parts[-1]} ({m_size_mb:.2f} MB)")
                print("-" * 60)

            # 🔥 核心魔法：如果是一个文件，显式地在文件流中跳过它的实际数据大小！
            # 这样 Python 就不用去解压和读取这个文件的图片数据，直接把指针拨到下一个文件头
            if member.isfile() and member.size > 0:
                # tar 格式内部是以 512 字节块对齐的，计算它占用的实际块大小
                blocks = (member.size + 511) // 512
                # 让底层文件流直接向前 skip 对应的字节数
                tf.fileobj.seek(blocks * 512, 1)

    print(f"采样结束，共快速定位了 {len(seen_directories)} 个不同的数据集子目录。")

In [6]:
find_out(TRAIN_TAR_PATH, max_preview=30)

正在对 42GB 压缩包进行【跨目录跳跃式】快速采样...
------------------------------------------------------------
New target [1/30]:
  V .../00328_Animalia_Arthropoda_Insecta_Coleoptera_Coccinellidae_Hippodamia_variegata/
  |____ Files /////: 1913d7ed-bad9-4702-9e09-ef24d7531c10.jpg (0.04 MB)
------------------------------------------------------------
New target [2/30]:
  V .../04827_Animalia_Chordata_Mammalia_Rodentia_Sciuridae_Geosciurus_inauris/
  |____ Files /////: 2846f2fe-df74-4818-a6dd-b966ff022305.jpg (0.13 MB)
------------------------------------------------------------
New target [3/30]:
  V .../09296_Plantae_Tracheophyta_Magnoliopsida_Rosales_Elaeagnaceae_Elaeagnus_angustifolia/
  |____ Files /////: dbd75380-200b-4b98-9d43-89de9df89d81.jpg (0.08 MB)
------------------------------------------------------------
New target [4/30]:
  V .../08333_Plantae_Tracheophyta_Magnoliopsida_Gentianales_Rubiaceae_Coprosma_rotundifolia/
  |____ Files /////: d3fdf76a-544e-4b26-bd22-0423fc97632a.jpg (0.10 MB)
-----

## Find the target of 500 ##

In [7]:
# seed set down
random.seed(RANDOM_SEED)

In [8]:
def stream_parse_annotation(json_tar_path):
    '''
    最开始，我使用了读取Tar然后逐个跑的方式，然后最开始是f.seek(0)重置，但由于文件是只读。
    所以seek的倒带操作会直接报错 io.UnsupportedOperation: seek

    所以现在是一次全部写入
    '''
    print(f"正在流式解析标注: {Path(json_tar_path).name}")
    
    imageid2path = {}
    cat2images = {}
    categories = []
    
    current_section = None
    current_item = None
    current_key = None
    
    with tarfile.open(json_tar_path, "r:gz") as tar:
        json_name = [n for n in tar.getnames() if n.endswith(".json") and ".." not in n][0]
        with tar.extractfile(json_name) as f:
            # 仅遍历一次 ijson.parse，完美绕过 seek(0)
            for prefix, event, value in tqdm(ijson.parse(f), desc="单次流式解析JSON"):
                # 检测进入哪个数组
                if prefix == "images.item" and event == "start_map":
                    current_section = "images"; current_item = {}
                    continue
                elif prefix == "annotations.item" and event == "start_map":
                    current_section = "annotations"; current_item = {}
                    continue
                elif prefix == "categories.item" and event == "start_map":
                    current_section = "categories"; current_item = {}
                    continue
                
                # 收集键值对
                if current_item is not None and event == "map_key":
                    current_key = value
                    continue
                if current_item is not None and event in {"string", "number", "boolean", "null"}:
                    current_item[current_key] = value
                    continue
                
                # 结束当前大括号，存入对应结构
                if current_item is not None and event == "end_map":
                    if current_section == "images":
                        imageid2path[current_item["id"]] = current_item["file_name"]
                    elif current_section == "annotations":
                        cat_id = current_item["category_id"]
                        img_id = current_item["image_id"]
                        # 注意：因为JSON里annotations可能在images前面，这里先存img_id，后面统一映射
                        if cat_id not in cat2images:
                            cat2images[cat_id] = []
                        cat2images[cat_id].append(img_id) 
                    elif current_section == "categories":
                        categories.append({
                            "category_id": current_item["id"],
                            "category_name": current_item["name"],
                            "common_name": current_item.get("common_name", ""),
                            "kingdom": current_item.get("kingdom", ""),
                            "supercategory": current_item.get("supercategory", "")
                        })
                    current_item = None

    # 后处理：把 cat2images 里的 image_id 转换为实际的 file_path
    print("正在进行内存索引对齐...")
    cat2images_path = {}
    for cat_id, img_ids in cat2images.items():
        cat2images_path[cat_id] = [imageid2path[mid] for mid in img_ids if mid in imageid2path]

    cat_df = pd.DataFrame(categories)
    print(f"解析完成：共 {len(cat_df)} 个物种，{sum(len(v) for v in cat2images.values())} 张图片")
    return cat_df, cat2images_path

In [9]:
def stratified_sample_classes(cat_df, num_classes=NUM_CLASSES):
    """
    按界（kingdom）分层随机抽样，保证植物/动物/真菌都按比例覆盖
    从而保证不会某一部分过于密集，稀疏或缺失
    返回选中的类别ID列表
    """
    print(f"\n分层抽样 {num_classes} 个物种...")
    selected = []
    kingdom_groups = cat_df.groupby("kingdom")["category_id"].unique()
    total = sum(len(g) for g in kingdom_groups)
    
    for kingdom, cats in kingdom_groups.items():
        ratio = len(cats) / total
        sample_n = int(ratio * num_classes)
        sampled = random.sample(sorted(cats.tolist()), sample_n)
        selected.extend(sampled)
        print(f"  {kingdom}: Total categories {len(cats)}, sampled {sample_n}")
        print(f"  {kingdom}: 总类数{len(cats)}，抽取{sample_n}类")
    
    # 补齐到正好num_classes
    remaining = num_classes - len(selected)
    if remaining > 0:
        all_cats = cat_df["category_id"].unique().tolist()
        pool = [c for c in all_cats if c not in selected]
        selected.extend(random.sample(sorted(pool), remaining))
    
    random.shuffle(selected)
    print(f"Sampling complete, total {len(selected)} classes selected")
    print(f"抽样完成，共选中 {len(selected)} 个物种")
    return selected


def split_train_val_set(cat2images, selected_classes, n_train=NUM_TRAIN_PER_CLASS, n_val=NUM_VAL_PER_CLASS):
    """
    按类别分层划分训练集和验证集
    返回：train_list, val_list，每项是 (file_path, label, category_id)
    """
    print("\n划分训练/验证集...")
    train_list = []
    val_list = []
    # /// CENTER OF ALL STUFF ///
    '''/// 这个label_map就是一切后续的源头，从而保证锚定在一个目标上 ///
        后面的 split_train_val_set 和 build_test_set 都是基于这个 label_map 来做的
        还有一个 save_annotation_files 则是把它写入到 label_mapping.json 里，供后续使用

        也就是说：一次运行里，label=17 在三个 split 里指向的都是同一个 category_id。
    '''
    label_map = {cid: idx for idx, cid in enumerate(sorted(selected_classes))}
    
    for cid in tqdm(selected_classes):
        imgs = cat2images[cid]
        random.shuffle(imgs)
        label = label_map[cid]
        
        for path in imgs[:n_train]:
            train_list.append((path, label, cid))
        for path in imgs[n_train:n_train+n_val]:
            val_list.append((path, label, cid))
    
    print(f"Split complete: Training set {len(train_list)} images, Validation set {len(val_list)} images")
    print(f"划分完成：训练集 {len(train_list)} 张，验证集 {len(val_list)} 张")
    return train_list, val_list, label_map


def build_test_set(val_cat2images, selected_classes, label_map):
    """构建测试集"""
    print("\n构建测试集...")
    test_list = []
    for cid in tqdm(selected_classes):
        imgs = val_cat2images[cid]
        label = label_map[cid]
        for path in imgs[:NUM_TEST_PER_CLASS]:
            test_list.append((path, label, cid))
    
    print(f"Test set construction complete: Total {len(test_list)} images")
    print(f"测试集构建完成：共 {len(test_list)} 张")
    return test_list


def stream_extract_to_tar(source_tar_path, target_tar, needed_paths, desc="提取中"):
    """
    【核心流式函数】从源tar包中筛选指定文件，写入目标tar包
    全程流式读写，内存仅占用当前单文件大小
    """
    needed_set = set(needed_paths)
    extracted_count = 0
    
    with tarfile.open(source_tar_path, "r:gz") as src_tar:
        # 遍历所有成员（只读取文件头，不读内容，内存极低）
        for member in tqdm(src_tar, desc=desc, unit="Files... "):
            # 只处理文件，跳过目录
            if not member.isfile():
                continue
            # 判断是否是需要的文件
            if member.name in needed_set:
                # 流式读取源文件，直接写入目标tar，不缓存全文件
                src_file = src_tar.extractfile(member)
                target_tar.addfile(member, src_file)
                extracted_count += 1
    
    print(f"  Extraction complete: Total {extracted_count} files extracted")
    print(f"  提取完成：共提取 {extracted_count} 个文件")
    return extracted_count


def save_annotation_files(train_list, val_list, test_list, cat_df, selected_classes, label_map):
    """生成标注CSV和映射文件，返回文件路径列表，后续写入tar包"""
    print("\n生成标注文件...")
    
    # 1. 数据集划分CSV
    train_df = pd.DataFrame(train_list, columns=["file_path", "label", "category_id"])
    val_df = pd.DataFrame(val_list, columns=["file_path", "label", "category_id"])
    test_df = pd.DataFrame(test_list, columns=["file_path", "label", "category_id"])
    
    # 补充物种名称
    cat_name_map = dict(zip(cat_df["category_id"], cat_df["category_name"]))
    for df in [train_df, val_df, test_df]:
        df["category_name"] = df["category_id"].map(cat_name_map)
    
    # 2. 类别列表
    selected_cat_df = cat_df[cat_df["category_id"].isin(selected_classes)].sort_values("category_id")
    
    # 3. 标签映射
    label_mapping = {
        "category_to_label": label_map,
        "label_to_category": {v: k for k, v in label_map.items()}
    }
    import json
    label_json = json.dumps(label_mapping, indent=2, ensure_ascii=False)
    
    # 保存到临时内存，直接写入tar（不落地到磁盘）
    files_to_add = [
        ("annotations/train.csv", train_df.to_csv(index=False).encode("utf-8")),
        ("annotations/val.csv", val_df.to_csv(index=False).encode("utf-8")),
        ("annotations/test.csv", test_df.to_csv(index=False).encode("utf-8")),
        ("annotations/class_list_500.csv", selected_cat_df.to_csv(index=False).encode("utf-8")),
        ("annotations/label_mapping.json", label_json.encode("utf-8")),
        ("README.md", f"""# tiny_inat_500.tar.gz #
- NUM_CLASSES: {NUM_CLASSES}
- NUM_TRAIN_PER_CLASS: {NUM_TRAIN_PER_CLASS}, total: {len(train_list)}
- NUM_VAL_PER_CLASS: {NUM_VAL_PER_CLASS}, total: {len(val_list)}
- NUM_TEST_PER_CLASS: {NUM_TEST_PER_CLASS}, total: {len(test_list)}
- RANDOM_SEED: {RANDOM_SEED}

- ~ This file format is: ~

tiny_inat_500.tar.gz
    |L annotations
    |   |F class_list_500.csv
    |   |F label_mapping.json
    |   |F test.csv
    |   |F train.csv
    |   |F val.csv
    |L train_mini
    |   |L 00006_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Aculepeira_ceropegia
    |   |   |F 0acbaf31-89b2-4768-8bd3-bf279a7b2f60.jpg
    |   |   |F ...
    |   |L 00079_Animalia_Arthropoda_Arachnida_Araneae_Salticidae_Helpis_minitabunda
    |   |L ...
    |L val
    |   |L 00006_Animalia_Arthropoda_Arachnida_Araneae_Araneidae_Aculepeira_ceropegia
    |   |   |F 4f539364-22bd-4215-a264-dbdae49330a8.jpg
    |   |   |F ...
    |   |L 00079_Animalia_Arthropoda_Arachnida_Araneae_Salticidae_Helpis_minitabunda
    |   |L ...
""".encode("utf-8"))
    ]
    return files_to_add

def line():
    print("-" * 60)

def main():
    line()
    print("Low-memory version of 500-class subset generate")
    print("低内存流式构建500类小子集")
    line()
    
    # 1. 解析训练集标注
    train_cat_df, train_cat2images = stream_parse_annotation(TRAIN_JSON_TAR_PATH)
    
    # 2. 分层抽样500类
    selected_classes = stratified_sample_classes(train_cat_df)
    # selected_classes = stratified_sample_classes(train_cat_df, 500)
    
    # 3. 划分训练/验证集
    train_list, val_list, label_map = split_train_val_set(train_cat2images, selected_classes)
    
    # 4. 解析测试集（val集）标注
    val_cat_df, val_cat2images = stream_parse_annotation(VAL_JSON_TAR_PATH)
    test_list = build_test_set(val_cat2images, selected_classes, label_map)
    
    # 5. 收集所有需要提取的图片路径
    train_paths = [p for p, _, _ in train_list]
    val_paths = [p for p, _, _ in val_list]
    test_paths = [p for p, _, _ in test_list]
    all_train_val_paths = train_paths + val_paths  # 都来自train_mini.tar.gz
    
    # 6. 流式构建输出tar包
    print("\nStartBuilding Output Tar...")
    print("\n开始构建输出tar包...")
    with tarfile.open(OUTPUT_TAR_PATH, "w:gz") as out_tar:
        # 提取训练+验证图片（都来自train_mini.tar.gz）
        print("extracting training and validation images...")
        print("提取训练集和验证集图片...")
        stream_extract_to_tar(TRAIN_TAR_PATH, out_tar, all_train_val_paths, desc="处理训练集tar")
        
        # 提取测试集图片（来自val.tar.gz）
        print("extracting test images...")
        print("提取测试集图片...")
        stream_extract_to_tar(VAL_TAR_PATH, out_tar, test_paths, desc="处理测试集tar")
        
        # 写入标注文件
        print("writing annotation files...")
        print("写入标注文件...")
        anno_files = save_annotation_files(
            train_list, val_list, test_list,
            train_cat_df, selected_classes, label_map
        )
        for path, content in anno_files:
            info = tarfile.TarInfo(name=path)
            info.size = len(content)
            import io
            out_tar.addfile(info, io.BytesIO(content))
    
    print("\n")
    line()
    print(f"^v^ %.*构建完成！输出文件：{OUTPUT_TAR_PATH}")
    print(f"       总大小约：{os.path.getsize(OUTPUT_TAR_PATH)/1024/1024/1024:.2f} GB")
    line()
    
    # 可选：同时解压到文件夹
    # print("\n正在解压到本地文件夹...")
    # OUTPUT_DIR.mkdir(exist_ok=True)
    # with tarfile.open(OUTPUT_TAR_PATH, "r:gz") as tar:
    #     tar.extractall(OUTPUT_DIR)
    # print(f"解压完成：{OUTPUT_DIR}")


if __name__ == "__main__":
    main()

------------------------------------------------------------
Low-memory version of 500-class subset generate
低内存流式构建500类小子集
------------------------------------------------------------
正在流式解析标注: train_mini.json.tar.gz


单次流式解析JSON: 15260085it [00:07, 2161091.27it/s]


正在进行内存索引对齐...
解析完成：共 10000 个物种，500000 张图片

分层抽样 500 个物种...
  Animalia: Total categories 5388, sampled 269
  Animalia: 总类数5388，抽取269类
  Fungi: Total categories 341, sampled 17
  Fungi: 总类数341，抽取17类
  Plantae: Total categories 4271, sampled 213
  Plantae: 总类数4271，抽取213类
Sampling complete, total 500 classes selected
抽样完成，共选中 500 个物种

划分训练/验证集...


100%|██████████| 500/500 [00:00<00:00, 28521.81it/s]

Split complete: Training set 20000 images, Validation set 5000 images
划分完成：训练集 20000 张，验证集 5000 张
正在流式解析标注: val.json.tar.gz


...

Test set construction complete: Total 5000 images
测试集构建完成：共 5000 张

StartBuilding Output Tar...

开始构建输出tar包...
extracting training and validation images...
提取训练集和验证集图片...


处理训练集tar: 510001Files...  [04:20, 1954.20Files... /s]


  Extraction complete: Total 25000 files extracted
  提取完成：共提取 25000 个文件
extracting test images...
提取测试集图片...


处理测试集tar: 110001Files...  [00:50, 2162.89Files... /s]


  Extraction complete: Total 5000 files extracted
  提取完成：共提取 5000 个文件
writing annotation files...
写入标注文件...

生成标注文件...


------------------------------------------------------------
^v^ %.*构建完成！输出文件：tiny_inat_500.tar.gz
       总大小约：2.51 GB
------------------------------------------------------------


In [10]:
def validate_output_tar(tar_path=OUTPUT_TAR_PATH):
    """
    检查输出 tar.gz 中的图片、CSV 和标签映射是否一致。
    
    Check the consistency for output tar.gz between images, CSV files, and label mappings.
    """
    tar_path = Path(tar_path)
    if not tar_path.exists():
        print(f"ERROR! no file found in: {tar_path}")
        return False

    print(f"Starting validation ~: {tar_path}")
    with tarfile.open(tar_path, "r:gz") as tar:
        members = tar.getnames()
        member_set = set(members)

        required_files = {
            "annotations/train.csv",
            "annotations/val.csv",
            "annotations/test.csv",
            "annotations/class_list_500.csv",
            "annotations/label_mapping.json",
            "README.md",
        }
        missing_required = sorted(required_files - member_set)

        if missing_required:
            raise AssertionError(f"BAD File! Missing required files: {missing_required}")

        train_df = pd.read_csv(tar.extractfile("annotations/train.csv"))
        val_df = pd.read_csv(tar.extractfile("annotations/val.csv"))
        test_df = pd.read_csv(tar.extractfile("annotations/test.csv"))
        class_df = pd.read_csv(tar.extractfile("annotations/class_list_500.csv"))
        label_map = json.load(tar.extractfile("annotations/label_mapping.json"))

        class_ids = set(class_df["category_id"].astype(str))
        label_keys = set(label_map["category_to_label"].keys())
        if label_keys != class_ids:
            raise AssertionError("!!! label_mapping.json && class_list_500.csv having different category_ids")

        combined = pd.concat([train_df, val_df, test_df], ignore_index=True)
        if combined["label"].isna().any():
            raise AssertionError("!!! train/val/test having EMPTY label")

        file_names = set()
        for frame_name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
            if frame["file_path"].isna().any():
                raise AssertionError(f"!!! {frame_name} having EMPTY file_path")
            file_names.update(frame["file_path"].astype(str).tolist())

        missing_images = sorted(name for name in file_names if name not in member_set)
        if missing_images:
            raise AssertionError(
                f"!!!There are {len(missing_images)} file_paths in the CSV files that are not found in the tar archive,"
                f" e.g.: {missing_images[:10]}"
            )

        duplicated_rows = combined.duplicated(subset=["file_path", "label", "category_id"]).sum()
        if duplicated_rows:
            print(f"!!! ERROR: detected multiple repeated rows: {duplicated_rows}")

        valid_labels = {int(k) for k in label_map["label_to_category"].keys()}
        csv_labels = set(combined["label"].astype(int).tolist())
        if csv_labels - valid_labels:
            raise AssertionError(f"!!! ERROR: CSV contains undefined labels: {sorted(csv_labels - valid_labels)}")

    print("v Validation passed v")
    print(f"  data-train: {len(train_df)}")
    print(f"  data-val: {len(val_df)}")
    print(f"  data-test: {len(test_df)}")
    print(f"  Total categories: {len(class_df)}")
    print(f"  Total files: {len(file_names)}")
    return True
validate_output_tar()

Starting validation ~: tiny_inat_500.tar.gz
v Validation passed v
  data-train: 20000
  data-val: 5000
  data-test: 5000
  Total categories: 500
  Total files: 30000


True

## After all running ##

This tiny_inat_500.tar.gz MUST been moving manually to data/